In [1]:
import os
import pandas as pd
from tqdm import tqdm
import numpy as np

# Очистка данных

In [2]:
df = pd.DataFrame()

for i in tqdm(range(len(os.listdir('chunks')))):
    chunk_file = f'chunks/chunk{i}.parquet'

    chunk = pd.read_parquet(chunk_file)
    df = pd.concat([df, chunk], ignore_index=True)

  0%|          | 0/44 [00:00<?, ?it/s]

100%|██████████| 44/44 [01:33<00:00,  2.12s/it]


## Избавляемся от выбросов

In [3]:
df['trip_duration'] = df['ended_at'] - df['started_at']
df = df[(df['trip_duration'] > pd.Timedelta(minutes=0)) & (df['trip_duration'] < pd.Timedelta(days=1))]

In [4]:
Q1 = df['trip_duration'].quantile(0.25)
Q3 = df['trip_duration'].quantile(0.75)
IQR = Q3 - Q1

df = df[df['trip_duration'] < Q3 + 3 * IQR] # Q3 + 3 * IQR = 1 час 15 секунд
df = df[df['trip_duration'] > pd.Timedelta(minutes=2)]

ставя такой порог (1 час 15 секунд), я потенцаинльно теряю данные о поездках по тарифу "Day Pass". но в общей выборке поездки такой длительности почти не представлены, из-за чего отличить их от простых выбросов (некорректный сбор данных) не представляется возможным.

In [5]:
df[['start_lat', 'start_lng', 'end_lat', 'end_lng', 'birth_year']] = \
    df[['start_lat', 'start_lng', 'end_lat', 'end_lng', 'birth_year']].astype(np.float32)

In [6]:
LEFT_LOWER_BOUND = {'lng': -89.2, 'lat': 40.8}
RIGHT_UPPER_BOUND = {'lng': -86.1, 'lat': 42.9}

In [7]:
df = df[
    ((df['start_lat'] < RIGHT_UPPER_BOUND['lat']) | df['start_lat'].isna()) &
    ((df['start_lat'] > LEFT_LOWER_BOUND['lat']) | df['start_lat'].isna()) &
    ((df['start_lng'] < RIGHT_UPPER_BOUND['lng']) | df['start_lng'].isna()) &
    ((df['start_lng'] > LEFT_LOWER_BOUND['lng']) | df['start_lng'].isna()) &
    ((df['end_lat'] < RIGHT_UPPER_BOUND['lat']) | df['end_lat'].isna()) &
    ((df['end_lat'] > LEFT_LOWER_BOUND['lat']) | df['end_lat'].isna()) &
    ((df['end_lng'] < RIGHT_UPPER_BOUND['lng']) | df['end_lng'].isna()) &
    ((df['end_lng'] > LEFT_LOWER_BOUND['lng']) | df['end_lng'].isna())
]

In [8]:
df = df[((df['birth_year'] > 1950) & ((df['started_at'].dt.year - df['birth_year']) >= 10)) | (df['birth_year'].isna())]

## Избавляемся от дубликатов

In [9]:
mask_to_drop = df.duplicated(subset=['started_at', 'ended_at', 'user_type',
                                     'start_station_name', 'end_station_name', 'rideable_type'])

In [10]:
df = df[~mask_to_drop]

## Приводим к двум .parquet файлам

In [11]:
before_2020 = df[df['started_at'] < pd.Timestamp(year=2020, month=1, day=1)].copy()
after_2020 = df[df['started_at'] >= pd.Timestamp(year=2020, month=1, day=1)].copy()

In [12]:
before_2020 = before_2020.drop(columns=['start_lat', 'start_lng', 'end_lat', 'end_lng', 'rideable_type'])
after_2020 = after_2020.drop(columns=['bike_id', 'gender', 'birth_year'])

In [13]:
before_2020.to_parquet('before_2020.parquet', index=False)
after_2020.to_parquet('after_2020.parquet', index=False)